In [5]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]  # go up from notebooks/
sys.path.append(str(ROOT))

In [ ]:
from config import SEED, N_LIST, METHODS, RESULTS_PATH
from models.cvae import train_cvae_on_arrays, sample_cvae_dataset
from models.bootstrap import sample_bootstrap
from models.gmm import sample_gmm
import numpy as np
import pandas as pd

# Loading Data
1. Breast cancer
2. Pima Indians Diabetes
3. Somethng else 2

#### 1. Breast Cancer

| Dataset | Category | n | p | Class 0 | Class 1 | Source | Notes |
|---------|----------|---|---|--------|--------|--------|-------|
| Breast Cancer | clinical_tabular | 569 | 30 | 212 | 357 | sklearn | moderate n, moderate p, numeric only |

In [ ]:
from loaders import load_breast
data = load_breast()

X = data["X"]
y = data["y"]
feature_names = data["feature_names"]

print(data["dataset"], data["category"])
print(X.shape, y.shape)
print(np.bincount(y))

breast_cancer clinical_tabular
(569, 30) (569,)
[212 357]


# Helper Functions
- Stratify Sampler

In [8]:
import numpy as np

def strat_samp(idx0, idx1, n0, n1, rng):
    a = rng.choice(idx0, size=n0, replace=False)
    b = rng.choice(idx1, size=n1, replace=False)
    return np.concatenate([a, b])

In [9]:
def stratified_subsample(X, y, n0, n1, seed=42):
    rng = np.random.default_rng(seed)

    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]

    idx = strat_samp(idx0, idx1, n0, n1, rng)
    rng.shuffle(idx)

    return X[idx], y[idx], idx

In [10]:
X_small, y_small, idx_small = stratified_subsample(X, y, n0=23, n1=68, seed=42)
print(X_small.shape)
print(np.bincount(y_small))

(91, 30)
[23 68]


- Train & sample CVAE

In [ ]:

best_state = train_cvae_on_arrays(
    X_small,
    y_small,
    seed=42,
    z_dim=16,
    hidden=128,
    beta=0.5,
    lr=1e-3,
    epochs=200,
    batch_size=32,
)

X_syn, y_syn = sample_cvae_dataset(
    best_state,
    n0=23,
    n1=68,
    seed=42
)

print(X_syn.shape)
print(np.bincount(y_syn))

Epoch  50 | val loss=8.9776 recon=7.1583 kl=3.6384
Epoch 100 | val loss=8.3683 recon=6.2910 kl=4.1546
Epoch 150 | val loss=7.3591 recon=5.1392 kl=4.4398
Epoch 200 | val loss=7.4349 recon=5.1087 kl=4.6525
(91, 30)
[23 68]


# Number shit

In [20]:
from metrics import evaluate_all

metrics = evaluate_all(X_real=X_small, X_syn=X_syn, seed=42)
print(metrics)

{'rf_auc': 0.6944444444444444, 'rf_sep': 0.6944444444444444, 'corr_mean_abs_diff': np.float64(0.08238319451070887), 'corr_max_abs_diff': np.float64(0.3565390292915948), 'pval_mean': np.float64(0.37860928753771134), 'pval_median': np.float64(0.2774363953399403), 'prop_significant': np.float64(0.1)}


### Results row

In [22]:
row = {
    "dataset": data["dataset"],
    "category": data["category"],
    "subgroup": "full",
    "n0": int((y_small == 0).sum()),
    "n1": int((y_small == 1).sum()),
    "n_total": len(y_small),
    "method": "cvae",
    "seed": 42,
    "rf_auc": metrics["rf_auc"],
    "corr_mean_abs_diff": metrics["corr_mean_abs_diff"],
    "prop_significant": metrics["prop_significant"],
}

df = pd.DataFrame([row])
display(df)

,dataset,category,subgroup,n0,n1,n_total,method,seed,rf_auc,corr_mean_abs_diff,prop_significant
0,breast_cancer,clinical_tabular,full,23,68,91,cvae,42,0.694444,0.082383,0.1
